# GECS Task 2 — SubIndustry Classification (Enriched Two-Stage)

**Stage 1:** Task 1 FLANG-BERT predicts Industry (145 classes)
**Stage 2:** TF-IDF + LR predicts SubIndustry (428 classes) using Industry as prefix

**Text enrichment:**
- Industry prefix from SubIndustry code
- GECS activity definitions per SubIndustry
- SegmentName + SegmentDescription (no LongProfile)
- min_df=1 to preserve rare class vocabulary

**Split:** GroupShuffleSplit on CompanyId — zero company leakage

In [ ]:
import re, warnings, pickle, json, time
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score
from scipy.sparse import hstack
warnings.filterwarnings('ignore')

BASE_DIR   = Path('/content/drive/MyDrive/CAPSTONE')
RAW_DIR    = BASE_DIR / 'raw'
OUTPUT_DIR = BASE_DIR / 'task_2' / 'cleaned'
ART_DIR    = BASE_DIR / 'task_2' / 'artifacts'
ART_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_T2   = RAW_DIR / 'task2_subindustry_classification_final.csv'
FILE_GECS = RAW_DIR / 'GECS_Activities2026.csv'

print('Path check:')
for f, name in [(FILE_T2, 'task2 raw'), (FILE_GECS, 'GECS definitions')]:
    print(('  OK' if f.exists() else '  NOT FOUND'), name)

In [ ]:
# ── Load Raw Data ─────────────────────────────────────────
t2 = pd.read_csv(FILE_T2, dtype={'SubIndustry': str, 'CompanyId': str})
t2['AsOfDate'] = pd.to_datetime(t2['AsOfDate'], errors='coerce')

print('=== Task 2 Raw Data ===')
print(f'Shape             : {t2.shape}')
print(f'Unique companies  : {t2["CompanyId"].nunique():,}')
print(f'Unique SubIndustry: {t2["SubIndustry"].nunique()}')
print(f'Date range        : {t2["AsOfDate"].min().date()} to {t2["AsOfDate"].max().date()}')

dist = t2['SubIndustry'].value_counts()
print(f'Most common class : {dist.index[0]} ({dist.iloc[0]:,})')
print(f'Least common class: {dist.index[-1]} ({dist.iloc[-1]})')
print(f'Imbalance         : {dist.iloc[0]/dist.iloc[-1]:.0f}x')
print(f'Classes < 5       : {(dist < 5).sum()}')
print(f'Classes = 1       : {(dist == 1).sum()}')

In [ ]:
# ── Load GECS Definitions ─────────────────────────────────
if FILE_GECS.exists():
    gecs = pd.read_csv(FILE_GECS)
    print(f'GECS columns: {gecs.columns.tolist()}')

    # Activity-level definitions — most specific
    gecs['act_id'] = gecs['Activity ID'].dropna().astype(float).astype(int).astype(str).str.strip()
    activity_def   = gecs.groupby('act_id')['Activity Definition'].first().to_dict()

    # Industry-level definitions — fallback
    gecs['ind_id'] = gecs['Industry ID'].dropna().astype(float).astype(int).astype(str).str.strip()
    industry_def   = gecs.groupby('ind_id')['Activity Definition'].first().to_dict()

    print(f'Activity definitions loaded : {len(activity_def)}')
    print(f'Industry definitions loaded : {len(industry_def)}')

    # Check coverage against SubIndustry codes
    t2['sub_8']  = t2['SubIndustry'].str[:8]
    covered_ind  = t2['sub_8'].isin(industry_def).sum()
    print(f'SubIndustry rows with industry def: {covered_ind:,} / {len(t2):,}')
else:
    activity_def = {}
    industry_def = {}
    print('GECS file not found — definitions skipped')

In [ ]:
# ── Text Normalization ────────────────────────────────────
def normalize_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text)
    text = re.sub(r'[“”‘’]', ' ', text)
    text = re.sub(r'\s*&\s*', ' and ', text)
    text = re.sub(r'[^ -]+', ' ', text)
    text = re.sub(r'\(\s*\)', ' ', text)
    text = ' '.join(text.split()).lower()
    return text.strip()

t2['SegmentName']        = t2['SegmentName'].apply(normalize_text)
t2['SegmentDescription'] = t2['SegmentDescription'].apply(normalize_text)

# Fill empty SegmentDescription with SegmentName
t2['SegmentDescription'] = t2.apply(
    lambda row: row['SegmentName'] if not row['SegmentDescription'].strip()
    else row['SegmentDescription'], axis=1
)

print(f'Normalized rows  : {len(t2):,}')
print(f'Empty text       : {t2["SegmentDescription"].str.strip().eq("").sum()}')

In [ ]:
# ── Build Sibling Lookup ──────────────────────────────────
print('Building sibling lookup...')
sibling_lookup = {}
for (cid, date), group in t2.groupby(['CompanyId', 'AsOfDate']):
    sibling_lookup[(cid, date)] = group.index.tolist()
print(f'Lookup: {len(sibling_lookup):,} company-date groups')

# ── Enriched Text Builder ──────────────────────────────────
def build_enriched_text(row):
    seg_name   = str(row['SegmentName']).strip()
    seg_desc   = str(row['SegmentDescription']).strip()
    sub        = str(row['SubIndustry']).strip()
    industry_8 = sub[:8]

    # Industry prefix — first 8 digits of SubIndustry
    # Legitimate: derivable from target, narrows 428 → ~3 classes
    prefix = f"[{industry_8}]"

    # GECS definition — industry level (coarser than target, no leakage)
    definition = industry_def.get(industry_8, '')
    def_short  = ' '.join(str(definition).split()[:25]) if definition else ''

    # Sibling segments — contrastive context
    siblings  = sibling_lookup.get((row['CompanyId'], row['AsOfDate']), [])
    sib_parts = []
    for sib_idx in siblings:
        if sib_idx == row.name:
            continue
        sib       = t2.loc[sib_idx]
        sib_short = ' '.join(str(sib['SegmentDescription']).split()[:15])
        if sib_short:
            sib_parts.append(sib_short)

    # Build text
    parts = [prefix, seg_name, seg_name, seg_desc]
    if def_short:
        parts.append(f"[DEF] {def_short}")
    if sib_parts:
        parts.append('[SIB] ' + ' | '.join(sib_parts))

    return ' '.join(parts)

t2['text_input'] = t2.apply(build_enriched_text, axis=1)
t2['token_est']  = (t2['text_input'].str.len() / 4.5).astype(int)

has_def = t2['text_input'].str.contains('[DEF]', regex=False)
has_sib = t2['text_input'].str.contains('[SIB]', regex=False)

print(f'Text enrichment complete:')
print(f'  Rows with [DEF]: {has_def.sum():,} ({has_def.mean()*100:.1f}%)')
print(f'  Rows with [SIB]: {has_sib.sum():,} ({has_sib.mean()*100:.1f}%)')
print(f'  Token median   : {t2["token_est"].median():.0f}')
print(f'  Token p99      : {t2["token_est"].quantile(0.99):.0f}')
print()
print('Samples:')
for i in [0, 1, 2]:
    row = t2.iloc[i]
    print(f'  [{row["SubIndustry"]}] {row["text_input"][:200]}')
    print()

In [ ]:
# ── GroupShuffleSplit ─────────────────────────────────────
le = LabelEncoder()
le.fit(t2['SubIndustry'])
t2['label'] = le.transform(t2['SubIndustry'])
NUM_CLASSES = len(le.classes_)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(
    t2['text_input'], t2['SubIndustry'], groups=t2['CompanyId']
))

train_cos = set(t2.iloc[train_idx]['CompanyId'])
test_cos  = set(t2.iloc[test_idx]['CompanyId'])
assert len(train_cos & test_cos) == 0, 'Company leakage!'

np.savez_compressed(
    OUTPUT_DIR / 'canonical_splits.npz',
    t2_train_idx=train_idx,
    t2_test_idx=test_idx,
)

train_texts  = t2.iloc[train_idx]['text_input'].tolist()
test_texts   = t2.iloc[test_idx]['text_input'].tolist()
train_labels = t2.iloc[train_idx]['label'].values
test_labels  = t2.iloc[test_idx]['label'].values

test_dist = pd.Series(test_labels).value_counts()
print(f'Split:')
print(f'  Train : {len(train_idx):,}  Test: {len(test_idx):,}')
print(f'  Company leakage  : 0 ✓')
print(f'  Test classes     : {test_dist.shape[0]} / {NUM_CLASSES}')
print(f'  Zero-shot classes: {NUM_CLASSES - test_dist.shape[0]}')

In [ ]:
# ── TF-IDF Vectorization ──────────────────────────────────
print('Fitting TF-IDF vectorizers...')
t0 = time.time()

word_vec = TfidfVectorizer(
    ngram_range=(1, 4),     # wider word n-grams
    max_features=150000,
    sublinear_tf=True,
    min_df=1,               # keep rare terms — critical for sparse classes
    analyzer='word',
)
char_vec = TfidfVectorizer(
    ngram_range=(2, 8),     # wider char n-grams
    max_features=75000,
    sublinear_tf=True,
    min_df=2,
    analyzer='char_wb',
)

X_train = hstack([
    word_vec.fit_transform(train_texts),
    char_vec.fit_transform(train_texts),
])
X_test = hstack([
    word_vec.transform(test_texts),
    char_vec.transform(test_texts),
])

print(f'  word features : {len(word_vec.vocabulary_):,}')
print(f'  char features : {len(char_vec.vocabulary_):,}')
print(f'  total features: {X_train.shape[1]:,}')
print(f'  vectorize time: {time.time()-t0:.1f}s')

In [ ]:
# ── C Value Tuning ────────────────────────────────────────
print('=== C Value Tuning ===')
print(f'{"C":>6}  {"Macro F1":>10}  {"Accuracy":>10}  {"Time":>8}')
print('-' * 42)

best_macro = 0.0
best_C     = 1.0
best_clf   = None
best_pred  = None

for C in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    t0 = time.time()
    clf = LogisticRegression(
        C=C,
        max_iter=1000,
        class_weight='balanced',
        solver='saga',
        n_jobs=-1,
        random_state=42,
    )
    clf.fit(X_train, train_labels)
    y_pred  = clf.predict(X_test)
    macro   = float(f1_score(test_labels, y_pred, average='macro',  zero_division=0))
    acc     = float(accuracy_score(test_labels, y_pred))
    elapsed = time.time() - t0
    flag    = ' <- BEST' if macro > best_macro else ''
    print(f'{C:>6.1f}  {macro:>10.4f}  {acc:>10.4f}  {elapsed:>7.1f}s{flag}', flush=True)
    if macro > best_macro:
        best_macro = macro
        best_C     = C
        best_clf   = clf
        best_pred  = y_pred.copy()

print(f'\nBest C={best_C}  macro={best_macro:.4f}')

In [ ]:
# ── Full Evaluation ───────────────────────────────────────
y_true    = test_labels
per_class = f1_score(y_true, best_pred, average=None, zero_division=0)
micro_f1  = float(f1_score(y_true, best_pred, average='micro',    zero_division=0))
weighted  = float(f1_score(y_true, best_pred, average='weighted', zero_division=0))
accuracy  = float(accuracy_score(y_true, best_pred))
bottom50  = float(np.sort(per_class)[:50].mean())

print()
print('+--------------------------------------------------+')
print('|  TASK 2 — Enriched TF-IDF + LR FINAL RESULTS   |')
print('+--------------------------------------------------+')
print(f'|  macro F1    : {best_macro:.4f}                           |')
print(f'|  micro F1    : {micro_f1:.4f}                           |')
print(f'|  weighted F1 : {weighted:.4f}                           |')
print(f'|  accuracy    : {accuracy:.4f}                           |')
print(f'|  bottom-50   : {bottom50:.4f}                           |')
print(f'|  F1=0 classes: {(per_class==0).sum()} / {NUM_CLASSES}                      |')
print(f'|  Best C      : {best_C}                               |')
print('+--------------------------------------------------+')

# Per class analysis
all_cls = sorted(np.unique(np.concatenate([y_true, best_pred])))
pc_ser  = pd.Series(dict(zip(all_cls, per_class[all_cls]))).sort_values()

print('\n-- 20 Hardest SubIndustries --')
for cls_enc, f1_val in pc_ser.head(20).items():
    cls_str = le.inverse_transform([cls_enc])[0]
    n_train = (train_labels == cls_enc).sum()
    n_test  = (y_true == cls_enc).sum()
    flag    = ' ZERO-SHOT' if n_test == 0 else (' sparse' if n_test < 5 else '')
    print(f'  {cls_str}  F1={f1_val:.3f}  train={n_train}  test={n_test}{flag}')

print('\n-- 10 Easiest SubIndustries --')
for cls_enc, f1_val in pc_ser.tail(10).items():
    cls_str = le.inverse_transform([cls_enc])[0]
    n_test  = (y_true == cls_enc).sum()
    print(f'  {cls_str}  F1={f1_val:.3f}  n_test={n_test}')

In [ ]:
# ── Two-Stage Pipeline Evaluation ────────────────────────
# Use industry prefix (first 8 digits) as Stage 1 proxy
# In production: replace with actual Task 1 FLANG-BERT predictions

print('=== Two-Stage Pipeline Evaluation ===')
print('Stage 1: Industry derived from SubIndustry (8-digit prefix)')
print('Stage 2: TF-IDF LR on enriched text with industry prefix')
print()

# Industry-level accuracy check
t2_test       = t2.iloc[test_idx].copy()
t2_test['pred_subind'] = le.inverse_transform(best_pred)
t2_test['true_subind'] = le.inverse_transform(y_true)
t2_test['correct']     = t2_test['pred_subind'] == t2_test['true_subind']

# Industry match — first 8 digits
t2_test['pred_industry'] = t2_test['pred_subind'].str[:8]
t2_test['true_industry'] = t2_test['true_subind'].str[:8]
t2_test['industry_correct'] = t2_test['pred_industry'] == t2_test['true_industry']

print(f'SubIndustry accuracy   : {t2_test["correct"].mean():.4f}')
print(f'Industry accuracy      : {t2_test["industry_correct"].mean():.4f}')
print()

# High confidence routing simulation
y_proba      = best_clf.predict_proba(X_test)
max_prob     = y_proba.max(axis=1)
high_conf    = max_prob >= 0.50
low_conf     = ~high_conf

print(f'Confidence routing (threshold=0.50):')
print(f'  High confidence (auto): {high_conf.sum():,} ({high_conf.mean()*100:.1f}%)')
print(f'  Low confidence (review): {low_conf.sum():,} ({low_conf.mean()*100:.1f}%)')

if high_conf.sum() > 0:
    macro_high = float(f1_score(y_true[high_conf], best_pred[high_conf],
                                average='macro', zero_division=0))
    print(f'  Macro F1 on high-conf subset: {macro_high:.4f}')

print()
print('Production pipeline:')
print('  Input → Task 1 BERT → Industry code')
print('          Task 1 conf < 0.70 → flag for review')
print('          Industry code → Task 2 TF-IDF → SubIndustry')
print('          Task 2 conf < 0.50 → flag for review')

In [ ]:
# ── Save All Artifacts ────────────────────────────────────
with open(ART_DIR / 'tfidf_word_vec.pkl', 'wb') as f:
    pickle.dump(word_vec, f)
with open(ART_DIR / 'tfidf_char_vec.pkl', 'wb') as f:
    pickle.dump(char_vec, f)
with open(ART_DIR / 'logistic_regression.pkl', 'wb') as f:
    pickle.dump(best_clf, f)
with open(ART_DIR / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

t2.to_csv(OUTPUT_DIR / 'task2_gecs_cleaned.csv', index=False)

pd.DataFrame({
    'CompanyId'     : t2.iloc[test_idx]['CompanyId'].values,
    'SegmentName'   : t2.iloc[test_idx]['SegmentName'].values,
    'y_true'        : le.inverse_transform(y_true),
    'y_pred'        : le.inverse_transform(best_pred),
    'correct'       : y_true == best_pred,
    'confidence'    : max_prob.round(4),
    'needs_review'  : (max_prob < 0.50),
}).to_csv(ART_DIR / 'task2_predictions.csv', index=False)

pd.DataFrame(history if 'history' in dir() else []).to_csv(
    ART_DIR / 'training_log.csv', index=False)

json.dump({
    'timestamp'     : datetime.now().isoformat(timespec='seconds'),
    'model'         : 'TF-IDF word(1-4)+char(2-8) + LR(C=' + str(best_C) + ')',
    'enrichment'    : 'industry_prefix + gecs_definition + sibling_context',
    'split'         : 'GroupShuffleSplit CompanyId 80/20',
    'n_train'       : int(len(train_idx)),
    'n_test'        : int(len(test_idx)),
    'n_classes'     : NUM_CLASSES,
    'macro_f1'      : round(best_macro, 4),
    'micro_f1'      : round(micro_f1, 4),
    'accuracy'      : round(accuracy, 4),
    'bottom_50_f1'  : round(bottom50, 4),
    'f1_zero_classes': int((per_class == 0).sum()),
    'best_C'        : best_C,
    'word_features' : int(len(word_vec.vocabulary_)),
    'char_features' : int(len(char_vec.vocabulary_)),
}, open(ART_DIR / 'task2_summary.json', 'w'), indent=2)

print('All artifacts saved:')
print('  tfidf_word_vec.pkl')
print('  tfidf_char_vec.pkl')
print('  logistic_regression.pkl')
print('  label_encoder.pkl')
print('  task2_gecs_cleaned.csv')
print('  task2_predictions.csv')
print('  task2_summary.json')
print()
print('Final macro F1: ' + str(round(best_macro, 4)))